# 02 - Preprocessing demo and exploratory data analysis (Phases 4 and 6)

**Before running:** the pipeline must have been executed once:
```
python -m src.data.combine_datasets
python -m src.data.preprocess
python -m src.data.split_dataset
```
Sections: 1 Setup | 2 The two preprocessing pipelines | 3 Load the splits | 4 Class distribution | 5 Language distribution |
6 Source distribution | 7 Text length | 8 XLM-R token length (optional) | 9 Examples | 10 Class imbalance and class weights | 11 Summary

> Warning: sections 2 and 9 print real comments that can contain abusive language. **Clear all outputs before committing to GitHub.**

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid")

from src.utils.config import (TRAIN_FILE, VAL_FILE, TEST_FILE, ZERO_SHOT_FILE, EXTERNAL_FILE,
                              FIGURES_DIR, METRICS_DIR, LABEL_NAMES)
from src.data.io_utils import read_processed_csv
from src.data.preprocess import clean_for_xlmr, clean_for_tfidf
FIGURES_DIR.mkdir(parents=True, exist_ok=True); METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 2. The two preprocessing pipelines (Phase 4)
The **same** raw comment is cleaned twice:
* **XLM-R** (`text` column): minimal. Keeps capital letters, emojis, hashtags and punctuation, because the transformer uses them as context.
* **TF-IDF** (`text_tfidf` column): classical normalisation (lower-case, punctuation removed, repeated letters squeezed).

The examples below are harmless sentences written for this demo.

In [ ]:
demo = [
    "I HATE this!!! 😡 <b>so bad</b> @bob_1, see https://t.co/xyz #Tamil &amp; more",
    "sooooo goooood ❤️❤️❤️",
    "<user> said <url> lol",
    "हिंदी वाक्य।। बहुत अच्छा!!!",
    "இது ஒரு சோதனை  \n\n பதிவு!!",
]
rows = [{"raw": r, "XLM-R text": clean_for_xlmr(r), "TF-IDF text": clean_for_tfidf(r)} for r in demo]
pd.DataFrame(rows)

## 3. Load the splits (Phase 5 output)

In [ ]:
train, val, test, zero = (read_processed_csv(p) for p in (TRAIN_FILE, VAL_FILE, TEST_FILE, ZERO_SHOT_FILE))
ext = read_processed_csv(EXTERNAL_FILE)
sets = {"train": train, "val": val, "test": test, "zero_shot": zero}
for k, v in sets.items():
    print(f"{k:<10}{len(v):>9,} rows")
print(f"{'external':<10}{len(ext):>9,} rows (IndicGuard - not analysed further here, reserved for final evaluation)")
all_df = pd.concat([v.assign(set=k) for k, v in sets.items()], ignore_index=True)
all_df["class"] = all_df["label"].map(LABEL_NAMES)
all_df.head(3)

## 4. Class distribution (Hate vs Non-Hate)

In [ ]:
tab = all_df.pivot_table(index="set", columns="class", values="text", aggfunc="count", fill_value=0)
tab = tab.reindex(["train", "val", "test", "zero_shot"])
tab["total"] = tab.sum(axis=1)
tab["hate_%"] = (100 * tab["Hate/Abusive"] / tab["total"]).round(1)
display(tab)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
tab[["Non-Hate", "Hate/Abusive"]].plot(kind="bar", stacked=True, ax=axes[0], color=["#4C72B0", "#C44E52"], rot=0)
axes[0].set_title("Class counts per set"); axes[0].set_ylabel("rows")
tr = train.assign(c=train["label"].map(LABEL_NAMES)).groupby(["language", "c"]).size().unstack(fill_value=0)
(100 * tr.div(tr.sum(axis=1), axis=0))[["Non-Hate", "Hate/Abusive"]].plot(kind="bar", stacked=True, ax=axes[1], color=["#4C72B0", "#C44E52"], rot=0)
axes[1].set_title("Training set: class share per language (%)"); axes[1].set_ylabel("%")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "eda_class_distribution.png", dpi=150); plt.show()

## 5. Language distribution

In [ ]:
lang = all_df.pivot_table(index="language", columns="set", values="text", aggfunc="count", fill_value=0)
lang = lang[["train", "val", "test", "zero_shot"]]
lang["total"] = lang.sum(axis=1)
display(lang)
ax = lang[["train", "val", "test", "zero_shot"]].plot(kind="bar", stacked=True, figsize=(9, 4), rot=0)
ax.set_title("Rows per language and set"); ax.set_ylabel("rows")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "eda_language_distribution.png", dpi=150); plt.show()

## 6. Dataset source distribution

In [ ]:
src = all_df.pivot_table(index="source", columns="set", values="text", aggfunc="count", fill_value=0)
display(src)
print("Share of the TRAINING set by source (%):")
print((100 * train["source"].value_counts(normalize=True)).round(1).to_string())

## 7. Sentence-length distribution
Characters and words per language (training set). This also tells us a sensible `max_length` for XLM-R later.

In [ ]:
d = train.copy()
d["chars"] = d["text"].str.len()
d["words"] = d["text"].str.split().str.len()
display(d.groupby("language")[["chars", "words"]].describe().round(1).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=d, x="language", y="words", ax=axes[0], showfliers=False); axes[0].set_title("Words per comment (outliers hidden)")
for lg, g in d.groupby("language"):
    sns.kdeplot(g["chars"].clip(upper=500), ax=axes[1], label=lg, fill=True, alpha=0.2, common_norm=False)
axes[1].set_title("Characters per comment (clipped at 500)"); axes[1].legend(); axes[1].set_xlabel("characters")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "eda_text_length.png", dpi=150); plt.show()
print("95th percentile of words per language:"); print(d.groupby("language")["words"].quantile(0.95).to_string())

## 8. XLM-R token length (optional)
Needs the `transformers` package and an internet connection once (small download of the tokenizer).
If it is not installed yet, the cell prints a message - that is fine, we install it in the next phase.

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
    sample = train.sample(min(4000, len(train)), random_state=42)
    sample["tokens"] = [len(x) for x in tok(sample["text"].tolist(), add_special_tokens=True)["input_ids"]]
    display(sample.groupby("language")["tokens"].describe(percentiles=[.5, .9, .95, .99]).round(1))
    print("-> a max_length of about", int(sample["tokens"].quantile(0.99)), "tokens covers 99% of comments")
except Exception as e:
    print("Skipped (", type(e).__name__, ") - install later with: pip install transformers")

## 9. Examples
Not random extremes: for every language and class we show 3 comments of **typical length** (between the 25th and 75th percentile),
chosen with a fixed random seed so the selection is reproducible.

In [ ]:
rng = 42
for lg in sorted(train["language"].unique()):
    for lab in (0, 1):
        g = train[(train["language"] == lg) & (train["label"] == lab)]
        n = g["text"].str.len()
        typical = g[(n >= n.quantile(0.25)) & (n <= n.quantile(0.75))]
        print(f"\n=== {lg} | label {lab} = {LABEL_NAMES[lab]} ({len(g):,} rows in train) ===")
        for t in typical["text"].sample(min(3, len(typical)), random_state=rng):
            print("  -", t[:140])

## 10. Class imbalance and class weights
We measure the imbalance for the whole training set and for every language.
**Decision rule used in this project:** if the minority class is below ~40% we use a **class-weighted loss** in XLM-R training
(weights saved in `results/metrics/class_weights.json`). We do not oversample first, because duplicating rows makes a transformer memorise them.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
def weights(y):
    cls = np.array(sorted(y.unique())); w = compute_class_weight("balanced", classes=cls, y=y)
    return {int(c): round(float(x), 4) for c, x in zip(cls, w)}

rows = []
for lg, g in train.groupby("language"):
    n1, n0 = int((g["label"] == 1).sum()), int((g["label"] == 0).sum())
    rows.append({"group": lg, "non_hate": n0, "hate": n1, "hate_%": round(100 * n1 / (n0 + n1), 1),
                 "majority/minority": round(max(n0, n1) / min(n0, n1), 2)})
n1, n0 = int((train["label"] == 1).sum()), int((train["label"] == 0).sum())
rows.append({"group": "ALL (train)", "non_hate": n0, "hate": n1, "hate_%": round(100 * n1 / (n0 + n1), 1),
             "majority/minority": round(max(n0, n1) / min(n0, n1), 2)})
imb = pd.DataFrame(rows); display(imb)

w = weights(train["label"])
print("Class weights for the weighted loss (class 0 = Non-Hate, class 1 = Hate/Abusive):", w)
minority_share = min(n0, n1) / (n0 + n1)
print("Minority share: %.1f%% ->" % (100 * minority_share),
      "weighted loss recommended" if minority_share < 0.40 else "classes are close to balanced; weights will be near 1 (still harmless to use)")
with open(METRICS_DIR / "class_weights.json", "w") as f:
    json.dump({"class_weights_train": w, "imbalance_table": imb.to_dict(orient="records")}, f, indent=2)

## 11. Summary
Write down (for your report) what the tables above show: number of rows per set, hate share, which language is smallest,
typical length, and the class weights. Then **Kernel -> Restart, Clear All Outputs** before committing.